<a href="https://colab.research.google.com/github/bayanasar/membraneclaw/blob/main/experiments/notebooks/toy-mdp/05_ppo_clipping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — PPO: Probability Ratios and Clipping

Notebook 04 ended on a dilemma. REINFORCE's gradient is only valid at the
policy that collected the data, so:

- **small steps** are safe but waste every sample (one update per batch, then
  the data is stale and must be thrown away);
- **large steps** are fast until the policy jumps somewhere its data does not
  justify — and a softmax policy that collapses onto a bad action stops
  sampling the alternatives it would need in order to recover.

We measured that: `alpha=0.02` was reliable, `alpha>=0.5` produced runs ending
around `-12`.

**Proximal Policy Optimization** resolves this by changing what we optimise.
Instead of one gradient step per batch, it takes *many* — while explicitly
refusing to move far from the policy that gathered the data. Two ideas do the
work: a **probability ratio**, and **clipping**.

This notebook builds both from scratch and, importantly, shows what clipping
does *and does not* protect against.

In [ ]:
# --- environment from notebook 01, repeated so this notebook stands alone ---
from __future__ import annotations

from enum import IntEnum
from typing import NamedTuple

import numpy as np

class State(IntEnum):
    NO_INFO = 0           # nothing checked yet
    SALINITY_CHECKED = 1  # feed salinity known
    FOULING_CHECKED = 2   # fouling indicators known
    BOTH_CHECKED = 3      # both kinds of evidence in hand
    SUCCESS = 4           # problem solved (terminal)
    FAILURE = 5           # wrong or unsafe fix submitted (terminal)


class Action(IntEnum):
    CHECK_SALINITY = 0
    CHECK_FOULING = 1
    RUN_SIMULATION = 2
    SUBMIT_DIRECTLY = 3


N_STATES, N_ACTIONS = len(State), len(Action)
TERMINAL_STATES = frozenset({State.SUCCESS, State.FAILURE})
NONTERMINAL = [s for s in State if s not in TERMINAL_STATES]


def is_terminal(state) -> bool:
    return State(state) in TERMINAL_STATES


COST_CHECK = -0.5          # first look at a piece of evidence
COST_REPEAT_CHECK = -1.0   # re-checking something already known: pure waste
REWARD_SIM_SUCCESS = 9.0   # +10 outcome, minus the -1 implicit cost of simulating
REWARD_SIM_FAILURE = -11.0
REWARD_SUBMIT_SUCCESS = 10.0
REWARD_SUBMIT_FAILURE = -10.0

SIM_SUCCESS_PROB = {
    State.NO_INFO: 0.15,
    State.SALINITY_CHECKED: 0.55,
    State.FOULING_CHECKED: 0.45,
    State.BOTH_CHECKED: 0.95,
}
SUBMIT_SUCCESS_PROB = {
    State.NO_INFO: 0.05,
    State.SALINITY_CHECKED: 0.35,
    State.FOULING_CHECKED: 0.25,
    State.BOTH_CHECKED: 0.75,
}


class Transition(NamedTuple):
    prob: float
    next_state: State
    reward: float


# Evidence held in each non-terminal state, used to work out where a check lands.
_EVIDENCE = {
    State.NO_INFO: frozenset(),
    State.SALINITY_CHECKED: frozenset({Action.CHECK_SALINITY}),
    State.FOULING_CHECKED: frozenset({Action.CHECK_FOULING}),
    State.BOTH_CHECKED: frozenset({Action.CHECK_SALINITY, Action.CHECK_FOULING}),
}
_STATE_BY_EVIDENCE = {ev: st for st, ev in _EVIDENCE.items()}


def transitions(state, action) -> tuple[Transition, ...]:
    # Every outcome of taking `action` in `state`, probabilities summing to 1.
    state, action = State(state), Action(action)

    if is_terminal(state):
        return (Transition(1.0, state, 0.0),)

    if action in (Action.CHECK_SALINITY, Action.CHECK_FOULING):
        already_known = action in _EVIDENCE[state]
        next_state = (
            state if already_known
            else _STATE_BY_EVIDENCE[_EVIDENCE[state] | {action}]
        )
        reward = COST_REPEAT_CHECK if already_known else COST_CHECK
        return (Transition(1.0, next_state, reward),)

    if action is Action.RUN_SIMULATION:
        p = SIM_SUCCESS_PROB[state]
        return (
            Transition(p, State.SUCCESS, REWARD_SIM_SUCCESS),
            Transition(1.0 - p, State.FAILURE, REWARD_SIM_FAILURE),
        )

    p = SUBMIT_SUCCESS_PROB[state]
    return (
        Transition(p, State.SUCCESS, REWARD_SUBMIT_SUCCESS),
        Transition(1.0 - p, State.FAILURE, REWARD_SUBMIT_FAILURE),
    )

def transition_tables() -> tuple[np.ndarray, np.ndarray]:
    # Dense tables for exact methods: P[s, a, s'] and expected R[s, a].
    P = np.zeros((N_STATES, N_ACTIONS, N_STATES))
    R = np.zeros((N_STATES, N_ACTIONS))
    for s in State:
        for a in Action:
            for prob, next_state, reward in transitions(s, a):
                P[s, a, next_state] += prob
                R[s, a] += prob * reward
    return P, R


P, R = transition_tables()

def policy_matrices(pi, P, R):
    # Collapse an MDP + policy into an MRP: (P_pi [S,S], R_pi [S]).
    P_pi = np.einsum("sa,sat->st", pi, P)
    R_pi = np.einsum("sa,sa->s", pi, R)
    return P_pi, R_pi


def deterministic(action_of_state) -> np.ndarray:
    # Build a [S, A] policy matrix from a {state: action} mapping.
    pi = np.zeros((N_STATES, N_ACTIONS))
    for s in State:
        pi[s, action_of_state(s)] = 1.0
    return pi


def step(state, action, rng) -> tuple[State, float, bool]:
    # Sample one environment step: (next_state, reward, done).
    # Inverse-CDF sampling from a single uniform draw. The obvious
    # rng.choice(..., p=probs) is ~4x slower per call, which matters once the
    # later notebooks run hundreds of thousands of episodes.
    outcomes = transitions(state, action)
    if len(outcomes) == 1:
        only = outcomes[0]
        return only.next_state, only.reward, is_terminal(only.next_state)
    u, cumulative = rng.random(), 0.0
    for prob, next_state, reward in outcomes:
        cumulative += prob
        if u < cumulative:
            return next_state, reward, is_terminal(next_state)
    last = outcomes[-1]  # float-rounding fallback
    return last.next_state, last.reward, is_terminal(last.next_state)


def rollout(policy_fn, rng, max_steps=20):
    # Run one episode; return a list of (state, action, reward) triples.
    s, traj = State.NO_INFO, []
    for _ in range(max_steps):
        a = policy_fn(s, rng)
        ns, r, done = step(s, a, rng)
        traj.append((s, a, r))
        s = ns
        if done:
            break
    return traj


def show(traj):
    total = sum(r for _, _, r in traj)
    for s, a, r in traj:
        print(f"  {State(s).name:<18} --{Action(a).name:<16}--> {r:+.1f}")
    print(f"  total (undiscounted) = {total:+.1f}")
    return total

GAMMA = 0.95

V_STAR = np.array([6.245, 7.1, 7.1, 8.0, 0.0, 0.0])
OPTIMAL_ACTIONS = {
    State.NO_INFO: {Action.CHECK_SALINITY, Action.CHECK_FOULING},
    State.SALINITY_CHECKED: {Action.CHECK_FOULING},
    State.FOULING_CHECKED: {Action.CHECK_SALINITY},
    State.BOTH_CHECKED: {Action.RUN_SIMULATION},
}


def softmax(x, axis=-1):
    z = x - x.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


def policy_probs(theta):
    return softmax(theta, axis=1)


def grad_log_pi(theta, s, a):
    g = np.zeros((N_STATES, N_ACTIONS))
    g[s] = -policy_probs(theta)[s]
    g[s, a] += 1.0
    return g


def sample_action(theta, s, rng):
    p = policy_probs(theta)[s]
    return Action(int(np.searchsorted(np.cumsum(p), rng.random())))


def run_episode(theta, rng, max_steps=20):
    s, traj = State.NO_INFO, []
    for _ in range(max_steps):
        a = sample_action(theta, s, rng)
        ns, r, done = step(s, a, rng)
        traj.append((s, a, r))
        s = ns
        if done:
            break
    return traj


def returns_along(traj, gamma=GAMMA):
    G, out = 0.0, []
    for _, _, r in reversed(traj):
        G = r + gamma * G
        out.append(G)
    return list(reversed(out))


def n_optimal(theta):
    probs = policy_probs(theta)
    return sum(Action(probs[s].argmax()) in OPTIMAL_ACTIONS[s] for s in NONTERMINAL)


print("setup complete; target v*(NO_INFO) =", V_STAR[State.NO_INFO])

## The probability ratio

Collect data with a policy $\pi_{\theta_\text{old}}$, then consider a candidate
new policy $\pi_\theta$. For each recorded $(s, a)$, define

$$r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_\text{old}}(a_t \mid s_t)}$$

This is **importance sampling**: it re-weights data gathered under the old
policy to estimate the new policy's performance.

- $r_t = 1$ — the new policy is unchanged for this action
- $r_t > 1$ — the new policy makes this action *more* likely
- $r_t < 1$ — *less* likely

The surrogate objective becomes

$$L^{\text{CPI}}(\theta) = \mathbb{E}_t\big[r_t(\theta)\, \hat{A}_t\big]$$

where $\hat{A}_t$ is the advantage from notebook 04. Its gradient at
$\theta = \theta_\text{old}$ (where every $r_t = 1$) is **exactly the REINFORCE
gradient with a baseline** — so this is the same algorithm, rewritten in a form
that stays meaningful when $\theta$ moves away from $\theta_\text{old}$.

That rewrite is what buys multiple epochs on one batch. But taken alone it is
also dangerous, which is the point of the next section.

In [ ]:
def ratio(theta_new, theta_old, s, a):
    return policy_probs(theta_new)[s, a] / policy_probs(theta_old)[s, a]


theta_old = np.zeros((N_STATES, N_ACTIONS))     # uniform: every action p = 0.25
s0, a0 = State.NO_INFO, Action.CHECK_SALINITY

print(f"old policy: pi(a|s) = {policy_probs(theta_old)[s0, a0]:.4f}\n")
print(f"{'theta[s,a]':>11}{'new pi(a|s)':>13}{'ratio':>9}")
for bump in [-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0]:
    tn = theta_old.copy()
    tn[s0, a0] += bump
    print(f"{bump:>11.1f}{policy_probs(tn)[s0, a0]:>13.4f}{ratio(tn, theta_old, s0, a0):>9.3f}")

Small parameter changes give ratios near 1; large ones move the ratio far from
1. The ratio is a direct, interpretable measure of **how much the policy has
moved** on the data we actually collected — which is exactly the quantity we
want to control.

## Why the unclipped objective is dangerous

Suppose an action had a positive advantage. $L^{\text{CPI}} = r_t \hat{A}_t$
grows without bound as $r_t$ grows, so an optimiser told to maximise it will
happily push that action's probability toward 1.

But the advantage estimate came from a **handful of noisy samples**. Trusting it
that far is unjustified, and if it was wrong the policy has now destroyed its
own ability to find out.

## The clipped surrogate objective

PPO's fix is to remove the incentive to move far:

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t\Big[\min\big(r_t \hat{A}_t,\;
   \text{clip}(r_t, 1-\epsilon, 1+\epsilon)\,\hat{A}_t\big)\Big]$$

with $\epsilon$ typically `0.1`–`0.3`. The `min` of the clipped and unclipped
terms makes this **pessimistic**: it takes the lower of the two.

The behaviour splits by the sign of the advantage:

- $\hat{A}_t > 0$ (good action): objective stops rewarding increases past
  $r_t = 1+\epsilon$. Gradient goes to zero — no incentive to push further.
- $\hat{A}_t < 0$ (bad action): objective stops rewarding decreases past
  $r_t = 1-\epsilon$. Gradient goes to zero again.

Crucially the clipping is **one-sided in each case**. If a step has already
overshot in the *harmful* direction, the gradient still points back. Let us plot
it.

In [ ]:
def L_clip(r, A, eps=0.2):
    return np.minimum(r * A, np.clip(r, 1 - eps, 1 + eps) * A)


EPS = 0.2
rs = np.linspace(0.0, 2.5, 26)

print(f"clipped objective, epsilon = {EPS}")
print(f"{'ratio':>7}{'A=+1 unclipped':>16}{'A=+1 clipped':>14}"
      f"{'A=-1 unclipped':>16}{'A=-1 clipped':>14}")
for r in rs[::3]:
    print(f"{r:>7.2f}{r * 1.0:>16.3f}{L_clip(r, 1.0, EPS):>14.3f}"
          f"{r * -1.0:>16.3f}{L_clip(r, -1.0, EPS):>14.3f}")

### The same thing, drawn

In [ ]:
R_MAX = 2.5


def plot_ascii(A, eps=EPS, width=51, height=13):
    rs_ = np.linspace(0, R_MAX, width)
    vals = L_clip(rs_, A, eps)
    lo, hi = min(vals.min(), 0.0), max(vals.max(), 0.0)
    band = (hi - lo) / (2 * height)

    def col(r):                       # column index nearest ratio r
        return int(round(r / R_MAX * (width - 1)))

    lo_col, hi_col, one_col = col(1 - eps), col(1 + eps), col(1.0)

    print(f"\nL_CLIP for advantage A = {A:+.0f}   (epsilon = {eps})")
    for row in range(height, -1, -1):
        y = lo + (hi - lo) * row / height
        line = ""
        for i, v in enumerate(vals):
            if abs(v - y) <= band:
                line += "*"
            elif i in (lo_col, hi_col):
                line += ":"                       # clip boundaries
            elif i == one_col:
                line += "|"                       # r = 1
            elif abs(y) < band:
                line += "-"                       # y = 0 axis
            else:
                line += " "
        print(f"{y:>7.2f} {line}")

    axis = [" "] * width
    for r in [0.0, 0.5, 1.0, 1.5, 2.0, 2.5]:
        axis[col(r)] = "^"
    print(f"{'':>8}{''.join(axis)}")

    labels = [" "] * width
    for r, txt in [(0.0, "0"), (0.5, "0.5"), (1.0, "1.0"), (1.5, "1.5"),
                   (2.0, "2.0"), (2.5, "2.5")]:
        start = max(0, min(col(r) - len(txt) // 2, width - len(txt)))
        labels[start:start + len(txt)] = txt
    print(f"{'':>8}{''.join(labels)}   <- probability ratio r")
    print(f"{'':>8}':' marks the clip range [{1 - eps:.1f}, {1 + eps:.1f}]"
          f", '|' marks r = 1")


plot_ascii(+1.0)
plot_ascii(-1.0)

Read the `A = +1` plot first. The objective rises with the ratio until
$r = 1.2$, then goes **flat**. Beyond that point, increasing the action's
probability earns nothing — so gradient ascent has no reason to go there.

Now `A = -1`. The objective is flat to the left of $r = 0.8$ and *falls* as $r$
grows. For a bad action the optimiser is pushed to reduce probability, but the
reward for doing so stops at $r = 0.8$.

The asymmetry is deliberate: the flat region always sits on the side where the
update would move *too far in the direction the data suggests*. Movement back
toward the old policy is never clipped, so a policy that has drifted too far can
always come home.

In [ ]:
# The gradient of L_CLIP w.r.t. the ratio: zero exactly where the objective is flat.
# Note that clipping is SIGN-DEPENDENT: the same ratio can be clipped for a
# positive advantage and still active for a negative one.
print(f"{'ratio':>7}{'dL/dr (A>0)':>13}{'  '}{'dL/dr (A<0)':>13}")
for r in [0.5, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.6]:
    h = 1e-6
    g_pos = (L_clip(r + h, 1.0) - L_clip(r - h, 1.0)) / (2 * h)
    g_neg = (L_clip(r + h, -1.0) - L_clip(r - h, -1.0)) / (2 * h)
    tag_p = "clipped" if abs(g_pos) < 1e-6 else "active "
    tag_n = "clipped" if abs(g_neg) < 1e-6 else "active "
    print(f"{r:>7.2f}{g_pos:>13.3f} {tag_p}{g_neg:>13.3f} {tag_n}")

This table is the precise statement of what clipping does, and the
sign-dependence is the part worth reading carefully.

For a **positive** advantage the gradient is alive for all $r < 1.2$ and dies
above it: the update may raise a good action's probability, but only up to the
trust boundary. For a **negative** advantage the mirror holds — alive for all
$r > 0.8$, dead below.

Look at $r = 0.5$: **clipped for $A<0$, still active for $A>0$.** A policy that
has already driven some action's probability far down is not prevented from
raising it again. Only the direction that moves *further* from the old policy is
switched off. That one-sidedness is why clipping constrains without trapping.

## Implementing PPO

The full loop, small but complete:

1. **Collect** a batch of episodes with the current policy.
2. **Compute** returns and advantages $\hat{A}_t = G_t - V(s_t)$.
3. **Snapshot** $\theta_\text{old}$ and the action probabilities under it.
4. **Update** for several epochs on the *same* batch, maximising
   $L^{\text{CLIP}}$.
5. **Fit** the critic to the observed returns.

The gradient of $L^{\text{CLIP}}$ for one sample, when not clipped, is
$r_t \hat{A}_t \nabla_\theta \log \pi_\theta(a_t|s_t)$ — the familiar policy
gradient scaled by the ratio. When clipped, it is zero.

In [ ]:
def collect_batch(theta, n_episodes, rng, gamma=GAMMA):
    # Returns a flat list of (state, action, G) tuples.
    batch = []
    ep_returns = []
    for _ in range(n_episodes):
        traj = run_episode(theta, rng)
        Gs = returns_along(traj, gamma)
        ep_returns.append(Gs[0])
        for (s, a, _), G in zip(traj, Gs):
            batch.append((s, a, G))
    return batch, float(np.mean(ep_returns))


def ppo(n_iters, batch_episodes=30, epochs=8, alpha=0.05, beta=0.1,
        eps=0.2, rng=None, gamma=GAMMA, clip=True):
    # PPO with a learned value baseline. Returns (theta, V, curve, clip_fraction).
    theta = np.zeros((N_STATES, N_ACTIONS))
    V = np.zeros(N_STATES)
    curve, clipped_frac = [], []

    for _ in range(n_iters):
        batch, mean_ret = collect_batch(theta, batch_episodes, rng, gamma)
        curve.append(mean_ret)

        old_probs = policy_probs(theta).copy()      # snapshot pi_old
        advantages = [G - V[s] for s, _, G in batch]

        n_clipped = 0
        for _ in range(epochs):
            grad = np.zeros_like(theta)
            probs = policy_probs(theta)
            for (s, a, _), A in zip(batch, advantages):
                r = probs[s, a] / old_probs[s, a]
                if clip and ((A > 0 and r > 1 + eps) or (A < 0 and r < 1 - eps)):
                    n_clipped += 1
                    continue                        # flat region: zero gradient
                grad += r * A * grad_log_pi(theta, s, a)
            theta += alpha * grad / len(batch)

        clipped_frac.append(n_clipped / max(1, epochs * len(batch)))

        for s, _, G in batch:                       # critic update
            V[s] += beta * (G - V[s])

    return theta, V, np.array(curve), np.array(clipped_frac)

### Training

In [ ]:
rng = np.random.default_rng(0)
theta_ppo, V_ppo, curve_ppo, clipfrac = ppo(60, rng=rng)

print("PPO after 60 iterations (30 episodes each = 1,800 episodes):")
probs = policy_probs(theta_ppo)
for s in NONTERMINAL:
    best = Action(probs[s].argmax())
    ok = "yes" if best in OPTIMAL_ACTIONS[s] else "NO"
    print(f"  {s.name:<18}{best.name:<17}p={probs[s].max():.3f}   optimal: {ok}")

print(f"\nmean return, first 5 iters: {curve_ppo[:5].mean():+.3f}")
print(f"mean return, last  5 iters: {curve_ppo[-5:].mean():+.3f}")
print(f"optimal:                    {V_STAR[State.NO_INFO]:+.3f}")
print(f"\ncritic: {np.round(V_ppo[:4], 2)}")
print(f"v*:     {V_STAR[:4]}")

### How often does clipping actually engage?

The clip fraction tells you whether the constraint is binding. Early in training
the policy changes fast and clipping fires often; as it converges, updates get
small and clipping becomes rare.

In [ ]:
print("clip fraction by training stage:")
for lo, hi in [(0, 5), (5, 15), (15, 30), (30, 45), (45, 60)]:
    seg = clipfrac[lo:hi]
    bar = "#" * int(seg.mean() * 200)
    print(f"  iters {lo:>2}-{hi:<3} {seg.mean():>7.4f}  {bar}")

print("\nlearning curve:")
for i in range(0, len(curve_ppo), 6):
    bar = "#" * max(0, int((curve_ppo[i] + 12) * 2))
    print(f"  iter {i:>3}  {curve_ppo[i]:>+7.3f}  {bar}")

## Does clipping actually matter?

We have built the mechanism and shown its shape. The real question is whether it
changes outcomes — so let us test it the way notebook 04 taught: **many seeds,
and report what happens rather than what should happen.**

The comparison is the same PPO loop with clipping switched off, which reduces to
running plain importance-weighted policy gradient for several epochs on one
batch.

In [ ]:
def compare(alpha, seeds=8, n_iters=40):
    out = {}
    for clip in [True, False]:
        finals, opts, worst = [], [], []
        for sd in range(seeds):
            r = np.random.default_rng(300 + sd)
            th, _, c, _ = ppo(n_iters, alpha=alpha, rng=r, clip=clip)
            finals.append(c[-5:].mean())
            opts.append(n_optimal(th))
        out[clip] = (np.mean(finals), np.std(finals), min(finals), np.mean(opts))
    return out


print(f"{'alpha':>7}{'clip':>7}{'mean':>9}{'sd':>8}{'worst':>9}{'optimal':>10}")
for alpha in [0.05, 0.5, 2.0]:
    res = compare(alpha)
    for clip in [True, False]:
        m, sd, w, o = res[clip]
        print(f"{alpha:>7}{str(clip):>7}{m:>+9.3f}{sd:>8.3f}{w:>+9.2f}{o:>9.1f}/4")

The `sd` and `worst` columns tell the story.

At `alpha=0.05` clipping is irrelevant — the steps are too small for the
constraint to bind, and both versions land in the same place. Notice this is
also the regime where PPO looks like a pointless complication, which is why
comparisons run only at safe step sizes prove nothing.

At `alpha=2.0` the versions separate sharply. Clipped stays tight and
near-optimal with a small spread and a *worst seed still near optimal*.
Unclipped collapses: the mean falls, the standard deviation explodes past `5`,
and the worst seed ends deeply negative — a policy that committed to a
catastrophic action and could not recover, exactly the failure mode from
notebook 04.

Compare this to notebook 04's step-size table, where **every** setting above
`alpha=0.1` degraded badly. PPO holds together at a step size forty times larger
than plain REINFORCE could tolerate — because no matter how large the raw
gradient is, each sample's contribution is switched off once the policy has
moved past its trust region.

That is the honest characterisation of PPO: not an optimiser that cannot fail,
but one that is far less sensitive to a hyperparameter nobody can tune in
advance. That robustness, more than peak performance, is why it became a
default.

## Epochs per batch: the sample-efficiency payoff

The point of the ratio was to reuse a batch. More epochs extract more from the
same data — until the policy has moved so far that the data no longer describes
it, and clipping is what determines how gracefully that limit is approached.

In [ ]:
print(f"{'epochs':>7}{'episodes used':>15}{'mean return':>13}{'clip frac':>11}")
for ep_count in [1, 2, 4, 8, 16]:
    finals, cfs = [], []
    for sd in range(5):
        r = np.random.default_rng(400 + sd)
        th, _, c, cf = ppo(30, epochs=ep_count, alpha=0.05, rng=r)
        finals.append(c[-5:].mean())
        cfs.append(cf[-10:].mean())
    print(f"{ep_count:>7}{30 * 30:>15}{np.mean(finals):>+13.3f}{np.mean(cfs):>11.4f}")

The effect is dramatic, and it is the payoff the whole ratio machinery was for.
Every row spends the **same 900 episodes**. With 1 epoch the agent is still
losing money at the end of training; with 16 epochs it is near optimal. Same
data, better policy — purely from extracting more signal per sample.

This is why PPO suits expensive environments. When an episode means a robot
trial or a day of simulation, "how much can I learn from the batch I already
have" is the question that matters, and REINFORCE's answer is "one gradient step,
then throw it away".

Watch the clip fraction climb with the epoch count — `0.0000` at 1–4 epochs,
rising as passes accumulate. At low epoch counts the policy never drifts far
enough for clipping to engage at all, so those rows are plain importance-weighted
policy gradient; the constraint only starts doing work once the reuse is
aggressive enough to need it.

The clip fraction is a genuinely useful diagnostic in real runs. Near zero means
the constraint is idle and you could push harder; very high means most of your
gradient is being discarded and the step size or epoch count is too aggressive.

## Summary of the series

| Notebook | Idea |
| --- | --- |
| 01 | states, actions, transitions, rewards, terminal states |
| 02 | Bellman equations, value iteration, policy extraction |
| 03 | Monte Carlo policy evaluation from sampled episodes |
| 04 | REINFORCE, baselines, and a learned critic |
| **05 — this one** | PPO's probability ratio and clipping |

The arc, in one line each:

1. Write the problem as an MDP.
2. With the model, solve it exactly — Bellman backups, no sampling.
3. Without the model, estimate values by averaging sampled returns.
4. Skip values; push on the policy directly, and fight the resulting variance.
5. Reuse each batch safely by constraining how far the policy may move.

Everything here is tabular and tiny, which is the point: each mechanism is
visible in isolation. Real implementations swap the lookup tables for neural
networks and add machinery we deliberately left out — GAE for advantage
estimation, entropy bonuses, value-function clipping, minibatching, learning-rate
annealing, observation normalisation. None of that changes the five ideas above.

### Things worth trying

- Set `eps=0.05` or `eps=1.0` in `ppo()` and watch the clip fraction respond.
- Turn off the critic (`beta=0.0`) and see how much the baseline was helping.
- Change `SIM_SUCCESS_PROB[State.BOTH_CHECKED]` to `0.5` and re-run notebook
  02's value iteration — does gathering evidence still pay?
- Raise `COST_CHECK` past `-8.0` and confirm the policy flips to committing
  immediately, as notebook 02's sweep predicted.